<a href="https://colab.research.google.com/github/AntonioAgostini/C.V.-Final-Project-/blob/Giorgia's/CVFinalProject_2_modifiche1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 2: Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios**

#**Imports**


In [1]:
import os
import json
import random
import zipfile
import tarfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from google.colab import drive

!pip install -q tqdm requests
import requests
from tqdm.auto import tqdm

#**Globals**

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT_DIR = Path("/content")
DATA_DIR = ROOT_DIR / "RRDataset"
DOWNLOAD_DIR = ROOT_DIR / "downloads"
EXTRACT_DIR = ROOT_DIR / "RRDataset_extracted"
CHECKPOINT_DIR = ROOT_DIR / "checkpoints"

DOWNLOAD_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

RF_LABELS = {"real": 0, "fake": 1}
TRANS_LABELS = {
    "original": 0,
    "internet_transmitted": 1,
    "re_digitized": 2
}

Device: cuda
GPU: Tesla T4


# **Utils**

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def show_image(image_path):
    img = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def list_files_recursively(root_path, max_items=100):
    count = 0
    for root, dirs, files in os.walk(root_path):
        level = root.replace(str(root_path), "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
            count += 1
            if count >= max_items:
                print("\n[Output truncated]")
                return

In [4]:
def download_with_progress(url, output_path, chunk_size=1024*1024):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(output_path, "wb") as f, tqdm(
        desc=output_path.name,
        total=total_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    print(f"Download completato: {output_path}")

#**Data**

## **2. Data Preparation e Simulazione di Robustezza**

### **Scelte Implementative sulla Selezione del Subset**
Poiché l'hardware di Colab (Tesla T4) pone vincoli di memoria e tempo, utilizziamo il pacchetto `RRDataset_original_train_val.tar.gz` (2.16 GB), che contiene 3000 immagini native ad alta risoluzione suddivise equamente in `real` e `ai`.

Per estendere il dataset al secondo task multi-classe (le 3 trasformazioni) senza saturare lo storage locale, implementiamo una pipeline di **Data Augmentation Dinamica on-the-fly** all'interno del modulo `Dataset` di PyTorch. Moltiplichiamo virtualmente il dataset per 3:
1. **Classe 0 (Original):** Immagine nativa pulita.
2. **Classe 1 (Internet-Transmitted):** Simulazione dei canali social (ridimensionamento bilineare dimezzato e forte compressione JPEG con fattore di qualità Q=30).
3. **Classe 2 (Re-Digitized):** Simulazione di ridigitalizzazione tramite applicazione di un filtro di sfocatura Gaussiano (Gaussian Blur, r=1.5) per emulare la degradazione della lente e del sensore d'acquisizione.

Questo approccio garantisce un **perfetto bilanciamento delle classi** per entrambi i task, eliminando qualsiasi bias statistico iniziale.

### Download and Extract RRDataset
This cells downloads the RRDataset from Zenodo and extracts it into the local environment.

##Real file recovery from Zenodo

In [7]:
import urllib.request

ZENODO_RECORD = "14963880"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD}"

with urllib.request.urlopen(ZENODO_API) as response:
    record_meta = json.loads(response.read().decode())

files_in_record = record_meta.get("files", [])

print("File trovati nel record Zenodo:")
for i, f in enumerate(files_in_record):
    print(f"{i}: {f['key']}  |  {round(f.get('size', 0)/1e9, 2)} GB")

print("\nDettaglio completo dei file:\n")
for f in files_in_record:
    print("Nome file:", f["key"])
    print("Dimensione (GB):", round(f.get("size", 0)/1e9, 2))
    print("Link diretto:", f["links"]["self"])
    print("-" * 60)

File trovati nel record Zenodo:
0: RRDataset_original_train_val.tar.gz  |  2.16 GB
1: RRDataset_test.tar.gz  |  20.12 GB

Dettaglio completo dei file:

Nome file: RRDataset_original_train_val.tar.gz
Dimensione (GB): 2.16
Link diretto: https://zenodo.org/api/records/14963880/files/RRDataset_original_train_val.tar.gz/content
------------------------------------------------------------
Nome file: RRDataset_test.tar.gz
Dimensione (GB): 20.12
Link diretto: https://zenodo.org/api/records/14963880/files/RRDataset_test.tar.gz/content
------------------------------------------------------------


##Download Dataset

In [8]:
target_file = files_in_record[0]   # poi eventualmente cambiamo
target_url = target_file["links"]["self"]
target_name = target_file["key"]

dest_path = DOWNLOAD_DIR / target_name

if not dest_path.exists():
    print("Scaricamento:", target_name)
    download_with_progress(target_url, dest_path)
else:
    print("File già presente:", dest_path)

File già presente: /content/downloads/RRDataset_original_train_val.tar.gz


##Archive extraction

In [12]:
import hashlib

def calculate_md5(file_path):
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

archive_path = dest_path

if not archive_path.exists():
    print(f"Errore: Il file archivio non esiste: {archive_path}. Si prega di eseguire la cella di download (precedente a questa).")
elif str(archive_path).endswith(".zip"):
    try:
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        print("Archivio ZIP estratto con successo in:", EXTRACT_DIR)
    except Exception as e:
        print(f"Errore durante l'estrazione del file ZIP: {e}")
        print(f"Si prega di verificare l'integrità del file o di eliminarlo manualmente: {archive_path} e riprovare a scaricarlo.")
elif str(archive_path).endswith(".tar.gz"):
    print(f"Verifica integrità per {archive_path}...")
    # The 'target_file' variable should be available from a previous cell's execution
    expected_checksum_full = target_file["checksum"] # e.g., 'md5:2f4498c3690d8f4c7a30d2e41dd34500'
    expected_md5 = expected_checksum_full.split(":")[1]

    calculated_md5 = calculate_md5(archive_path)

    if calculated_md5 == expected_md5:
        print("Checksum MD5 verificato. Il file è integro. Procedo con l'estrazione.")
        try:
            with tarfile.open(archive_path, "r:gz") as tf:
                tf.extractall(EXTRACT_DIR)
            print("Archivio TAR.GZ estratto con successo in:", EXTRACT_DIR)
        except EOFError:
            print(f"EOFError durante l'estrazione di {archive_path}. Il file potrebbe essere parzialmente corrotto nonostante il checksum o ci sono problemi con il formato dell'archivio.")
            print(f"Si prega di eliminare manualmente il file corrotto: {archive_path} e rieseguire la cella di download per un nuovo tentativo.")
        except Exception as e:
            print(f"Errore durante l'estrazione del file TAR.GZ: {e}")
            print(f"Si prega di verificare l'integrità del file o di eliminarlo manualmente: {archive_path} e riprovare a scaricarlo.")
    else:
        print(f"Errore di integrità: Checksum MD5 non corrispondente per {archive_path}.")
        print(f"Calcolato: {calculated_md5}, Atteso: {expected_md5}")
        print(f"Il file è corrotto. Si prega di eliminarlo manualmente: {archive_path} e rieseguire la cella di download per scaricarlo nuovamente.")
        print("Skipping extraction due to file corruption.")
else:
    print(f"Tipo di archivio non supportato: {archive_path}. Supportati .zip e .tar.gz")

print("Prime cartelle trovate:")
# Ensure EXTRACT_DIR exists even if extraction failed, to prevent error on rglob
EXTRACT_DIR.mkdir(exist_ok=True)
for p in sorted(EXTRACT_DIR.rglob("*")):
    if p.is_dir():
        print(p)

Verifica integrità per /content/downloads/RRDataset_original_train_val.tar.gz...
Errore di integrità: Checksum MD5 non corrispondente per /content/downloads/RRDataset_original_train_val.tar.gz.
Calcolato: db2041200e7d57e76208d8f68492b893, Atteso: 2f4498c3690d8f4c7a30d2e41dd34500
Il file è corrotto. Si prega di eliminarlo manualmente: /content/downloads/RRDataset_original_train_val.tar.gz e rieseguire la cella di download per scaricarlo nuovamente.
Skipping extraction due to file corruption.
Prime cartelle trovate:
/content/RRDataset_extracted/RRDataset_original_train_val
/content/RRDataset_extracted/RRDataset_original_train_val/train
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai
/content/RRDataset_extracted/RRDataset_original_train_val/train/real
/content/RRDataset_extracted/RRDataset_original_train_val/val
/content/RRDataset_extracted/RRDataset_original_train_val/val/ai
/content/RRDataset_extracted/RRDataset_original_train_val/val/real


##Dataset structure inspection

In [13]:
#print("Struttura del dataset estratto:\n")
#list_files_recursively(EXTRACT_DIR, max_items=200)

##Image count

In [14]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_images = []
for p in EXTRACT_DIR.rglob("*"):
    if p.suffix.lower() in IMG_EXTENSIONS:
        all_images.append(p)

print("Numero totale immagini trovate:", len(all_images))

#for p in all_images[:20]:
#    print(p)

Numero totale immagini trovate: 1986


Implementazione della Pipline Dati Multi-Task

In [24]:
import io
from PIL import ImageFilter

class MultiTaskRRDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.split_dir = Path(root_dir) / "RRDataset_original_train_val" / split
        self.transform = transform
        self.image_paths = []
        self.rf_labels = []

        for label_name, label_idx in RF_LABELS.items():
            folder = self.split_dir / ("ai" if label_name == "fake" else "real")
            if folder.exists():
                for img_p in folder.glob("*"):
                    if img_p.suffix.lower() in IMG_EXTENSIONS:
                        self.image_paths.append(img_p)
                        self.rf_labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths) * 3

    def __getitem__(self, idx):
        # Aggiungi un contatore di tentativi per evitare loop infiniti con un dataset completamente corrotto
        max_attempts = 5
        for attempt in range(max_attempts):
            base_idx = idx % len(self.image_paths)
            trans_idx = idx // len(self.image_paths)

            img_path = self.image_paths[base_idx]
            rf_label = self.rf_labels[base_idx]

            try:
                img = Image.open(img_path).convert("RGB")

                # Applicazione dinamica delle alterazioni post-processing
                if trans_idx == 1:
                    img = img.resize((IMG_SIZE // 2, IMG_SIZE // 2), Image.Resampling.BILINEAR)
                    img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
                    buffer = io.BytesIO()
                    img.save(buffer, format="JPEG", quality=30)
                    buffer.seek(0)
                    img = Image.open(buffer).convert("RGB")
                elif trans_idx == 2:
                    img = img.resize((IMG_SIZE, IMG_SIZE))
                    img = img.filter(ImageFilter.GaussianBlur(radius=1.5))

                if self.transform:
                    img = self.transform(img)

                return img, torch.tensor(rf_label, dtype=torch.long), torch.tensor(trans_idx, dtype=torch.long)
            except OSError as e:
                print(f"Avviso: Immagine corrotta o troncata saltata: {img_path} - {e}. Tentativo {attempt + 1}/{max_attempts}")
                # Incrementa l'indice per tentare di caricare un'altra immagine
                idx = (idx + 1) % len(self)

        # Se tutti i tentativi falliscono, solleva un errore o restituisci un sample vuoto/dummy
        raise RuntimeError(f"Impossibile caricare un'immagine valida dopo {max_attempts} tentativi. Considera la pulizia del dataset.")

# Definizione dei Trasformatori PyTorch
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = MultiTaskRRDataset(EXTRACT_DIR, split="train", transform=train_transform)
val_dataset = MultiTaskRRDataset(EXTRACT_DIR, split="val", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Campioni totali di Addestramento: {len(train_dataset)}")
print(f"Campioni totali di Validazione: {len(val_dataset)}")

Campioni totali di Addestramento: 4458
Campioni totali di Validazione: 1500


#**Network**

## **3. Modello Unificato Multi-Task (Shared Backbone)**

### **Scelte Architetturali**
Per mappare simultaneamente feature eterogenee (il riconoscimento di pattern generativi richiede l'analisi di dettagli microscopici ad alta frequenza, mentre il tipo di alterazione richiede l'analisi globale della tessitura e della sfocatura), utilizziamo una **ResNet50** preaddestrata su ImageNet come **Backbone Condiviso**.

Rimuoviamo lo strato di classificazione nativo (`fc = nn.Identity()`) e colleghiamo due teste lineari parallele e indipendenti allo spazio delle feature estratte (vettore a 2048 dimensioni):
* **Head Real/Fake (`head_rf`):** Uno strato lineare fully-connected che mappa le feature in 2 classi (reale vs IA).
* **Head Transformation (`head_trans`):** Uno strato lineare parallelo che mappa le medesime feature in 3 classi (tipo di post-processing).

In [17]:
class MultiTaskModel(nn.Module):
    def __init__(self, backbone_name="resnet50", pretrained=True):
        super(MultiTaskModel, self).__init__()

        if backbone_name == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            self.backbone = models.resnet50(weights=weights)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            raise ValueError("Backbone non implementato.")

        # Head 1: Classificazione Binaria Real/Fake
        self.head_rf = nn.Linear(num_features, 2)

        # Head 2: Classificazione Multi-classe delle Alterazioni
        self.head_trans = nn.Linear(num_features, 3)

    def forward(self, x):
        # Condivisione delle feature di basso e alto livello
        features = self.backbone(x)

        # Inferenza parallela sui due task
        out_rf = self.head_rf(features)
        out_trans = self.head_trans(features)
        return out_rf, out_trans

model = MultiTaskModel(backbone_name="resnet50", pretrained=True).to(DEVICE)
print(f"Modello inizializzato con successo. Parametri totali: {count_parameters(model):,}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]


Modello inizializzato con successo. Parametri totali: 23,518,277


# **Train**

## **4. Funzione di Loss Congiunta e Addestramento**

La loss di ottimizzazione globale del modello viene definita come la combinazione lineare pesata delle singole Cross-Entropy Loss dei due task:

$$L_{total} = \alpha \cdot L_{RF} + \beta \cdot L_{Trans}$$

In questa configurazione iniziale di baseline impostiamo $\alpha = 1.0$ e $\beta = 1.0$, assegnando pari dignità algoritmica a entrambi gli obiettivi. L'ottimizzatore scelto è **AdamW** (con `learning_rate = 1e-4` e un leggero `weight_decay`), ideale per prevenire l'overfitting delle teste lineari durante il fine-tuning del backbone.

In [19]:
criterion_rf = nn.CrossEntropyLoss()
criterion_trans = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

ALPHA = 1.0  # Peso associato al Task Real/Fake
BETA = 1.0   # Peso associato al Task Post-Processing

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_rf, correct_trans = 0, 0
    total = 0

    for imgs, rf_labels, trans_labels in tqdm(dataloader, desc="Training"):
        imgs, rf_labels, trans_labels = imgs.to(device), rf_labels.to(device), trans_labels.to(device)

        optimizer.zero_grad()
        out_rf, out_trans = model(imgs)

        loss_rf = criterion_rf(out_rf, rf_labels)
        loss_trans = criterion_trans(out_trans, trans_labels)

        # Loss Multi-Task Combinata
        total_loss = (ALPHA * loss_rf) + (BETA * loss_trans)
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item() * imgs.size(0)

        _, pred_rf = torch.max(out_rf, 1)
        _, pred_trans = torch.max(out_trans, 1)

        correct_rf += (pred_rf == rf_labels).sum().item()
        correct_trans += (pred_trans == trans_labels).sum().item()
        total += imgs.size(0)

    return running_loss / total, correct_rf / total, correct_trans / total

def validate(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    all_rf_preds, all_rf_labels = [], []
    all_trans_preds, all_trans_labels = [], []

    with torch.no_grad():
        for imgs, rf_labels, trans_labels in dataloader:
            imgs, rf_labels, trans_labels = imgs.to(device), rf_labels.to(device), trans_labels.to(device)

            out_rf, out_trans = model(imgs)
            loss_rf = criterion_rf(out_rf, rf_labels)
            loss_trans = criterion_trans(out_trans, trans_labels)
            total_loss = (ALPHA * loss_rf) + (BETA * loss_trans)

            running_loss += total_loss.item() * imgs.size(0)

            _, pred_rf = torch.max(out_rf, 1)
            _, pred_trans = torch.max(out_trans, 1)

            all_rf_preds.extend(pred_rf.cpu().numpy())
            all_rf_labels.extend(rf_labels.cpu().numpy())
            all_trans_preds.extend(pred_trans.cpu().numpy())
            all_trans_labels.extend(trans_labels.cpu().numpy())

    total = len(dataloader.dataset)
    acc_rf = accuracy_score(all_rf_labels, all_rf_preds)
    acc_trans = accuracy_score(all_trans_labels, all_trans_preds)

    return running_loss / total, acc_rf, acc_trans, all_rf_preds, all_rf_labels, all_trans_preds, all_trans_labels

Esecuzione del Training Loop


In [ ]:
import io

history = {"train_loss": [], "val_loss": [], "val_acc_rf": [], "val_acc_trans": []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc_rf, train_acc_trans = train_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, val_acc_rf, val_acc_trans, _, _, _, _ = validate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc_rf"].append(val_acc_rf)
    history["val_acc_trans"].append(val_acc_trans)

    print(f"Epoca {epoch+1}/{NUM_EPOCHS} -> Loss Addestramento: {train_loss:.4f} | Loss Validazione: {val_loss:.4f}")
    print(f"Accuratezza R/F: {val_acc_rf:.4f} | Accuratezza Trasformazioni: {val_acc_trans:.4f}\n")

Training:   0%|          | 0/278 [00:00<?, ?it/s]

Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Epoca 1/10 -> Loss Addestramento: 0.2688 | Loss Validazione: 0.5576
Accuratezza R/F: 0.7753 | Accuratezza Trasformazioni: 0.9987



Training:   0%|          | 0/278 [00:00<?, ?it/s]

Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Epoca 2/10 -> Loss Addestramento: 0.1171 | Loss Validazione: 0.4577
Accuratezza R/F: 0.8407 | Accuratezza Trasformazioni: 0.9980



Training:   0%|          | 0/278 [00:00<?, ?it/s]

Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Epoca 3/10 -> Loss Addestramento: 0.0723 | Loss Validazione: 0.5683
Accuratezza R/F: 0.8207 | Accuratezza Trasformazioni: 0.9993



Training:   0%|          | 0/278 [00:00<?, ?it/s]

Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5
Epoca 4/10 -> Loss Addestramento: 0.0497 | Loss Validazione: 0.9234
Accuratezza R/F: 0.7893 | Accuratezza Trasformazioni: 0.9867



Training:   0%|          | 0/278 [00:00<?, ?it/s]

Avviso: Immagine corrotta o troncata saltata: /content/RRDataset_extracted/RRDataset_original_train_val/train/real/real_003200.jpg - image file is truncated (3 bytes not processed). Tentativo 1/5


#**Evaluation/Test**